# MODIS Full Coverage Analysis Over Iran

## Terra MOD021KM and Aqua MYD021KM Spatial Coverage Assessment

**Author:** Mona Fakhri  
**Field:** Remote Sensing Engineering

This notebook provides a reproducible workflow for searching MODIS Terra/Aqua
Level-1B granules, extracting swath geometries, calculating spatial coverage
over Iran, and generating GIS-based outputs.


## 1. Scientific Motivation

MODIS observations are acquired as wide satellite swaths. Depending on the
orbital geometry, a single acquisition may provide partial or complete
coverage of a region of interest.

This workflow identifies MODIS granules intersecting Iran and calculates the
percentage of Iranian territory covered by each observation.


## 2. Data Source

- Terra MOD021KM (MODIS Level-1B)
- Aqua MYD021KM (MODIS Level-1B)
- Provider: NASA Earthdata

Processing steps:

Earthdata API → Granule Metadata → Swath Geometry → Coverage Analysis → Map & Report


In [ ]:
from pathlib import Path
import os
import getpass

import earthaccess
import geopandas as gpd
import pandas as pd
import folium

from shapely.geometry import box, Polygon, mapping
from tqdm import tqdm
from openpyxl.utils import get_column_letter


In [ ]:
# Project configuration

PROJECT_DIR = Path("..")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

IRAN_SHP_PATH = DATA_DIR / "gadm41_IRN_0.shp"

OUTPUT_DIR.mkdir(exist_ok=True, parents=True)


## 3. NASA Earthdata Authentication

In [ ]:
def authenticate_earthdata():
    username = input("Earthdata username: ").strip()
    password = getpass.getpass("Earthdata password: ")

    os.environ["EARTHDATA_USERNAME"] = username
    os.environ["EARTHDATA_PASSWORD"] = password

    auth = earthaccess.login(strategy="environment")

    if auth.authenticated:
        print("Authentication successful")
        return auth

    raise RuntimeError("Authentication failed")

# auth = authenticate_earthdata()


## 4. Load Iran Boundary

In [ ]:
def load_iran_shapefile(path):
    iran = gpd.read_file(path)
    iran = iran.to_crs(epsg=4326)
    polygon = iran.union_all()
    return iran, polygon

# iran_gdf, iran_polygon = load_iran_shapefile(IRAN_SHP_PATH)


## 5. MODIS Granule Search

In [ ]:
def search_modis(product, temporal_range, bbox):
    return earthaccess.search_data(
        short_name=product,
        temporal=temporal_range,
        bounding_box=bbox
    )


# Example:
# terra = search_modis("MOD021KM", ("2018-01-01","2025-11-15"), (44,25,63.5,40))
# aqua  = search_modis("MYD021KM", ("2018-01-01","2025-11-15"), (44,25,63.5,40))


## 6. Metadata Extraction

In [ ]:
def extract_granule_metadata(granule):

    metadata = {}
    umm = granule.get("umm", {})

    short_name = umm.get("CollectionReference", {}).get("ShortName","")

    if "MYD" in short_name:
        metadata["platform"] = "Aqua"
    elif "MOD" in short_name:
        metadata["platform"] = "Terra"
    else:
        metadata["platform"] = "Unknown"

    temporal = umm.get("TemporalExtent",{}).get("RangeDateTime",{})
    metadata["start_time"] = temporal.get("BeginningDateTime")
    metadata["end_time"] = temporal.get("EndingDateTime")

    dg = umm.get("DataGranule",{})
    metadata["producer_granule_id"] = dg.get("ProducerGranuleId")

    geometry = (
        umm.get("SpatialExtent",{})
        .get("HorizontalSpatialDomain",{})
        .get("Geometry",{})
    )

    if "BoundingRectangles" in geometry:
        b = geometry["BoundingRectangles"][0]
        metadata["bbox"] = {
            "west": float(b["WestBoundingCoordinate"]),
            "south": float(b["SouthBoundingCoordinate"]),
            "east": float(b["EastBoundingCoordinate"]),
            "north": float(b["NorthBoundingCoordinate"])
        }

    return metadata


## 7. Geometry and Coverage Analysis

In [ ]:
def create_granule_polygon(metadata):

    if "bbox" in metadata:
        b = metadata["bbox"]
        return box(
            b["west"],
            b["south"],
            b["east"],
            b["north"]
        )

    return None


def calculate_coverage(granules, iran_polygon):

    results=[]

    for granule in tqdm(granules):

        meta = extract_granule_metadata(granule)
        poly = create_granule_polygon(meta)

        if poly and poly.intersects(iran_polygon):

            intersection = poly.intersection(iran_polygon)

            meta["coverage_percentage"] = round(
                intersection.area / iran_polygon.area * 100,
                2
            )

            meta["geometry"] = poly
            results.append(meta)

    return results


## 8. Interactive Visualization

In [ ]:
def coverage_color(cov):

    if cov >= 90:
        return "purple"
    elif cov >= 80:
        return "red"
    elif cov >= 70:
        return "orange"
    elif cov >= 50:
        return "yellow"
    else:
        return "blue"


def create_map(results, iran_gdf):

    m = folium.Map(
        location=[32,53],
        zoom_start=5
    )

    folium.GeoJson(iran_gdf).add_to(m)

    for item in results:

        folium.GeoJson(
            mapping(item["geometry"]),
            style_function=lambda x,
            c=coverage_color(item["coverage_percentage"]): {
                "color": c,
                "fillColor": c,
                "fillOpacity": 0.3
            }
        ).add_to(m)

    return m


## 9. Export Results

In [ ]:
def export_excel(results, output_file):

    df = pd.DataFrame(results)

    df = df.sort_values(
        "coverage_percentage",
        ascending=False
    )

    df.to_excel(
        output_file,
        index=False
    )


# Example:
# export_excel(results, OUTPUT_DIR/"MODIS_results.xlsx")


## 10. Summary

The workflow provides a reproducible framework for MODIS Terra/Aqua coverage
analysis over Iran using NASA Earthdata and geospatial processing.
